# Prefetch Assets

Interactive notebook for downloading **all** project datasets, RAG knowledge corpora,
and models. Assets are saved locally so you can inspect them before committing to DVC.

## Overview

| # | Section | Purpose | Source |
|---|---------|---------|--------|
| 1 | Training — Summarization | LoRA fine-tuning | `ccdv/arxiv-summarization` (HF) |
| 2 | Training — Code Generation | LoRA fine-tuning | `nvidia/OpenCodeInstruct` (HF) |
| 3 | Eval — Code | HumanEval benchmark | `openai/openai_humaneval` (HF) |
| 4 | Eval — QA | Natural Questions | `google-research-datasets/natural_questions` (HF) |
| 5 | Eval — QA | HotpotQA | `hotpot_qa` (HF) |
| 6 | Eval — Retrieval | MS MARCO | `ms_marco` (HF) |
| 7 | Eval — Retrieval | BEIR (subset) | `BeIR/*` (HF) |
| 8 | RAG — Chat | ArXiv papers | arXiv API |
| 9 | RAG — Code | PyTorch docs | Web scraping |
| 10 | Models | Base LLMs | HuggingFace Hub |

## How to use

1. Set `PROJECT_ROOT` to the root of your local clone.
2. Run the **Setup** cell.
3. Execute only the sections you need — each section is independent.
4. Inspect downloaded assets, then `dvc add` / `dvc push`.

> **Note:** Vector index building is done separately via
> `experiments/scripts/rag_data/build_vector_index.py` after the raw data is collected here.

---
## 0. Setup

In [1]:
from __future__ import annotations

import json
import time
from pathlib import Path
from urllib.parse import urljoin

import arxiv
import requests
from bs4 import BeautifulSoup
from datasets import DatasetDict, load_dataset, load_from_disk
from huggingface_hub import snapshot_download

/home/anton-m/Git/agent-042/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os

os.getcwd()

'/home/anton-m/Git/agent-042/notebooks'

In [5]:
# ---- Set the project root path for your machine ----
PROJECT_ROOT = Path("/home/jovyan")
PROJECT_ROOT = Path("/home/anton-m/Git/agent-042")

ASSETS_DIR = PROJECT_ROOT / "assets"

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
print(f"Project root : {PROJECT_ROOT}")
print(f"Assets dir   : {ASSETS_DIR}")

Project root : /home/anton-m/Git/agent-042
Assets dir   : /home/anton-m/Git/agent-042/assets


---
## 0.1 Helper functions

In [6]:
def save_hf_dataset(
    dataset_name: str,
    target_dir: Path,
    *,
    dataset_config: str | None = None,
    train_split: str = "train",
    val_split: str | None = "validation",
) -> Path:
    """Download a HuggingFace dataset and save it to disk as Arrow files.

    Args:
        dataset_name: HuggingFace dataset identifier.
        target_dir: Local directory to save the dataset.
        dataset_config: Dataset configuration / subset name.
        train_split: Train split descriptor (supports slicing, e.g. 'train[:1%]').
        val_split: Validation split descriptor. Set to None to skip.

    Returns:
        Path to the saved dataset directory.
    """
    target_dir.mkdir(parents=True, exist_ok=True)
    print(
        f"Downloading {dataset_name}" + (f" [{dataset_config}]" if dataset_config else "") + " ..."
    )

    splits = {}
    splits["train"] = load_dataset(dataset_name, dataset_config, split=train_split)
    if val_split is not None:
        splits["validation"] = load_dataset(dataset_name, dataset_config, split=val_split)

    ds = DatasetDict(splits)
    print(f"Saving to disk: {target_dir}")
    ds.save_to_disk(str(target_dir))
    return target_dir


def inspect_hf_dataset(target_dir: Path) -> DatasetDict:
    """Load a previously saved HF dataset from disk and print basic stats."""
    ds = load_from_disk(str(target_dir))
    print(ds)
    for split_name, split_ds in ds.items():
        print(f"  {split_name}: {len(split_ds)} rows, columns={split_ds.column_names}")
    return ds

---
# Part I — Training Datasets

---

## 1. Summarization — `ccdv/arxiv-summarization`

LoRA fine-tuning for the summarization task.
Maps `article → abstract`. Using a **1 % subset** by default to keep download fast.

In [ ]:
ARXIV_SUMM_DIR = ASSETS_DIR / "datasets" / "arxiv-summarization"

save_hf_dataset(
    "ccdv/arxiv-summarization",
    ARXIV_SUMM_DIR,
    dataset_config="document",
    train_split="train",
    val_split="validation",
)
print(f"\n✅ arxiv-summarization saved to: {ARXIV_SUMM_DIR}")

In [8]:
# Inspect
ds = inspect_hf_dataset(ARXIV_SUMM_DIR)
ds["train"][0]

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 203037
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6436
    })
})
  train: 203037 rows, columns=['article', 'abstract']
  validation: 6436 rows, columns=['article', 'abstract']


{'article': 'additive models @xcite provide an important family of models for semiparametric regression or classification . some reasons for the success of additive models are their increased flexibility when compared to linear or generalized linear models and their increased interpretability when compared to fully nonparametric models . it is well - known that good estimators in additive models are in general less prone to the curse of high dimensionality than good estimators in fully nonparametric models . many examples of such estimators belong to the large class of regularized kernel based methods over a reproducing kernel hilbert space @xmath0 , see e.g. @xcite . in the last years many interesting results on learning rates of regularized kernel based models for additive models have been published when the focus is on sparsity and when the classical least squares loss function is used , see e.g. @xcite , @xcite , @xcite , @xcite , @xcite , @xcite and the references therein . of cou

## 2. Code Generation — `nvidia/OpenCodeInstruct`

LoRA fine-tuning for code generation.
Filtered to **Python** examples. Using a small subset by default.

In [ ]:
CODE_INSTRUCT_DIR = ASSETS_DIR / "datasets" / "open-code-instruct"

save_hf_dataset(
    "nvidia/OpenCodeInstruct",
    CODE_INSTRUCT_DIR,
    train_split="train",
    val_split=None,  # single-split dataset
)
print(f"\n✅ OpenCodeInstruct saved to: {CODE_INSTRUCT_DIR}")

Generating train split: 100%|██████████| 5000000/5000000 [00:56<00:00, 88052.84 examples/s] 


Saving to disk: /home/anton-m/Git/agent-042/assets/datasets/open-code-instruct-01


Saving the dataset (39/39 shards): 100%|██████████| 5000000/5000000 [01:21<00:00, 61588.21 examples/s] 



✅ OpenCodeInstruct saved to: /home/anton-m/Git/agent-042/assets/datasets/open-code-instruct-01


In [10]:
# Inspect
ds = inspect_hf_dataset(CODE_INSTRUCT_DIR)
ds["train"][0]

DatasetDict({
    train: Dataset({
        features: ['id', 'input', 'output', 'domain', 'generation_algorithm', 'llm_judgement', 'unit_tests', 'tests_execution_status', 'average_test_score'],
        num_rows: 5000000
    })
})
  train: 5000000 rows, columns=['id', 'input', 'output', 'domain', 'generation_algorithm', 'llm_judgement', 'unit_tests', 'tests_execution_status', 'average_test_score']


{'id': '4d61a897c0f86467f5c504a62a5f7282',
 'input': 'You are given a list of `n` tasks, each represented as a tuple `(start, end)`, indicating the start and end times of the task. The tasks are sorted by their start times. Your goal is to determine the maximum number of non-overlapping tasks that can be selected. Two tasks are considered non-overlapping if the start time of one task is greater than or equal to the end time of the other.\n\n**Input:**\n- An integer `n` representing the number of tasks.\n- A list of `n` tuples, where each tuple `(start, end)` represents the start and end times of a task.\n\n**Output:**\n- An integer representing the maximum number of non-overlapping tasks that can be selected.\n\n**Constraints:**\n- `1 <= n <= 10^5`\n- `0 <= start < end <= 10^9`\n\n**Sample Input:**\n```\n3\n1 3\n2 5\n4 6\n```\n\n**Sample Output:**\n```\n2\n```',
 'output': '```python\ndef max_non_overlapping_tasks(tasks):\n    """\n    Returns the maximum number of non-overlapping task

---
# Part II — Evaluation Datasets

---

## 3. Code Eval — `openai/openai_humaneval`

Measures executable rate and test-pass rate of generated code.

In [12]:
HUMANEVAL_DIR = ASSETS_DIR / "datasets" / "humaneval"

save_hf_dataset(
    "openai/openai_humaneval",
    HUMANEVAL_DIR,
    train_split="test",  # HumanEval only has a "test" split
    val_split=None,
)
print(f"\n✅ HumanEval saved to: {HUMANEVAL_DIR}")

README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Saving to disk: /home/jovyan/assets/datasets/humaneval


Saving the dataset (0/1 shards):   0%|          | 0/164 [00:00<?, ? examples/s]


✅ HumanEval saved to: /home/jovyan/assets/datasets/humaneval


In [13]:
# Inspect
ds = inspect_hf_dataset(HUMANEVAL_DIR)
ds["train"][0]

DatasetDict({
    train: Dataset({
        features: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'],
        num_rows: 164
    })
})
  train: 164 rows, columns=['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point']


{'task_id': 'HumanEval/0',
 'prompt': 'from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    """ Check if in given list of numbers, are any two numbers closer to each other than\n    given threshold.\n    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\n    False\n    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\n    True\n    """\n',
 'canonical_solution': '    for idx, elem in enumerate(numbers):\n        for idx2, elem2 in enumerate(numbers):\n            if idx != idx2:\n                distance = abs(elem - elem2)\n                if distance < threshold:\n                    return True\n\n    return False\n',
 'test': "\n\nMETADATA = {\n    'author': 'jt',\n    'dataset': 'test'\n}\n\n\ndef check(candidate):\n    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True\n    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False\n    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.95) == True\n    assert 

## 4. QA Eval — Natural Questions

Used for evaluating answer relevance and correctness (LLM-as-judge).
Downloading a **1 % subset** of the validation split.

In [ ]:
NQ_DIR = ASSETS_DIR / "datasets" / "natural-questions-01"

save_hf_dataset(
    "google-research-datasets/natural_questions",
    NQ_DIR,
    dataset_config="default",
    train_split="train[:1%]",
    val_split="validation[:5%]",
)
print(f"\n✅ Natural Questions saved to: {NQ_DIR}")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

default/train-00000-of-00287.parquet:   0%|          | 0.00/192M [00:00<?, ?B/s]

default/train-00001-of-00287.parquet:   0%|          | 0.00/202M [00:00<?, ?B/s]

...

default/train-00095-of-00287.parquet:   0%|          | 0.00/202M [00:00<?, ?B/s]

default/train-00096-of-00287.parquet:   0%|          | 0.00/187M [00:00<?, ?B/s]

default/train-00097-of-00287.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

default/train-00098-of-00287.parquet:   0%|          | 0.00/187M [00:00<?, ?B/s]

default/train-00099-of-00287.parquet:   0%|          | 0.00/197M [00:00<?, ?B/s]

In [ ]:
# Inspect
ds = inspect_hf_dataset(NQ_DIR)
ds["validation"][0]

## 5. QA Eval — HotpotQA

Multi-hop QA evaluation. Using the `distractor` setting and a small subset.

In [ ]:
HOTPOTQA_DIR = ASSETS_DIR / "datasets" / "hotpotqa-01"

save_hf_dataset(
    "hotpot_qa",
    HOTPOTQA_DIR,
    dataset_config="distractor",
    train_split="train[:1%]",
    val_split="validation[:5%]",
)
print(f"\n✅ HotpotQA saved to: {HOTPOTQA_DIR}")

In [ ]:
# Inspect
ds = inspect_hf_dataset(HOTPOTQA_DIR)
ds["validation"][0]

## 6. Retrieval Eval — MS MARCO

Passage ranking benchmark. Used for Recall@k and nDCG@k evaluation.
Downloading a small validation subset.

In [ ]:
MSMARCO_DIR = ASSETS_DIR / "datasets" / "msmarco-01"

save_hf_dataset(
    "ms_marco",
    MSMARCO_DIR,
    dataset_config="v1.1",
    train_split="train[:1%]",
    val_split="validation[:5%]",
)
print(f"\n✅ MS MARCO saved to: {MSMARCO_DIR}")

In [ ]:
# Inspect
ds = inspect_hf_dataset(MSMARCO_DIR)
ds["validation"][0]

## 7. Retrieval Eval — BEIR

BEIR is a heterogeneous benchmark for information retrieval.
We download individual sub-datasets relevant to the project (SciFact, NFCorpus).
Add more subsets as needed.

In [ ]:
BEIR_SUBSETS = ["scifact", "nfcorpus"]

for subset in BEIR_SUBSETS:
    subset_dir = ASSETS_DIR / "datasets" / f"beir-{subset}"
    save_hf_dataset(
        f"BeIR/{subset}",
        subset_dir,
        train_split="corpus",  # BEIR splits: corpus, queries
        val_split=None,
    )
    print(f"✅ BEIR/{subset} saved to: {subset_dir}\n")

In [ ]:
# Inspect any BEIR subset
subset_dir = ASSETS_DIR / "datasets" / f"beir-{BEIR_SUBSETS[0]}"
ds = inspect_hf_dataset(subset_dir)
ds["train"][0]

---
# Part III — RAG Knowledge Corpora

Raw data for the RAG vector store. After downloading here, build the vector
index with `experiments/scripts/rag_data/build_vector_index.py`.

---

## 8. Chat RAG — ArXiv Papers

Downloads recent ML/DL papers (metadata + abstracts) from the arXiv API.
Default: **100 papers** from `cs.LG` and `cs.AI`.

In [ ]:
# ---- ArXiv configuration ----
ARXIV_CATEGORIES = ["cs.LG", "cs.AI"]
ARXIV_MAX_RESULTS = 100
ARXIV_OUTPUT_DIR = ASSETS_DIR / "rag_data" / "arxiv"

In [ ]:
def download_arxiv_papers(
    categories: list[str],
    max_results: int,
    output_dir: Path,
) -> list[dict]:
    """Download papers from arXiv and save metadata + abstracts to JSON.

    Args:
        categories: arXiv categories (e.g. ['cs.LG', 'cs.AI']).
        max_results: Maximum papers to fetch.
        output_dir: Directory to write arxiv_papers.json.

    Returns:
        List of paper metadata dicts.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    query = " OR ".join(f"cat:{cat}" for cat in categories)
    print(f"Searching arXiv: {query}  (max {max_results})")

    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )

    papers = []
    for i, result in enumerate(client.results(search), 1):
        papers.append(
            {
                "arxiv_id": result.entry_id.split("/")[-1],
                "title": result.title,
                "authors": [a.name for a in result.authors],
                "abstract": result.summary,
                "published": result.published.isoformat(),
                "updated": result.updated.isoformat(),
                "categories": result.categories,
                "primary_category": result.primary_category,
                "pdf_url": result.pdf_url,
            }
        )
        if i % 20 == 0:
            print(f"  {i} papers fetched ...")

    output_file = output_dir / "arxiv_papers.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(papers, f, indent=2, ensure_ascii=False)

    print(f"Downloaded {len(papers)} papers → {output_file}")
    return papers

In [ ]:
arxiv_papers = download_arxiv_papers(ARXIV_CATEGORIES, ARXIV_MAX_RESULTS, ARXIV_OUTPUT_DIR)
print(f"\n✅ ArXiv papers saved to: {ARXIV_OUTPUT_DIR}")

In [ ]:
# Inspect
print(f"Total papers: {len(arxiv_papers)}")
print(f"Example title: {arxiv_papers[0]['title']}")
print(f"Categories: {arxiv_papers[0]['categories']}")
print(f"Abstract (first 300 chars): {arxiv_papers[0]['abstract'][:300]}...")

## 9. Code RAG — PyTorch Documentation

Scrapes a curated set of PyTorch documentation pages (core API, tutorials).
Saved as JSON for later chunking and indexing.

In [ ]:
# ---- PyTorch docs configuration ----
PYTORCH_BASE_URL = "https://pytorch.org/docs/stable/"
PYTORCH_OUTPUT_DIR = ASSETS_DIR / "rag_data" / "pytorch_docs"

# Core API pages to scrape (extend as needed)
PYTORCH_PAGES = [
    "generated/torch.nn.Module.html",
    "generated/torch.Tensor.html",
    "generated/torch.nn.Linear.html",
    "generated/torch.nn.Conv2d.html",
    "generated/torch.nn.functional.relu.html",
    "generated/torch.optim.Adam.html",
    "generated/torch.optim.SGD.html",
    "generated/torch.nn.CrossEntropyLoss.html",
    "generated/torch.nn.MSELoss.html",
    "generated/torch.autograd.backward.html",
    "tensors.html",
    "autograd.html",
    "nn.html",
    "optim.html",
    "torch.html",
]

In [ ]:
def scrape_pytorch_doc_page(url: str) -> dict:
    """Scrape a single PyTorch documentation page."""
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "lxml")

    title_tag = soup.find("h1")
    title_text = title_tag.get_text(strip=True) if title_tag else "Untitled"

    content_div = soup.find("div", {"role": "main"}) or soup.find("article")
    if content_div:
        for tag in content_div.find_all(["nav", "footer", "script", "style"]):
            tag.decompose()
        content = content_div.get_text(separator="\n", strip=True)
    else:
        content = ""

    code_blocks = soup.find_all("code") or soup.find_all("pre")
    code_examples = [b.get_text(strip=True) for b in code_blocks[:10]]

    return {
        "url": url,
        "title": title_text,
        "content": content,
        "code_examples": code_examples,
        "scraped_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }


def collect_pytorch_docs(
    base_url: str,
    page_list: list[str],
    output_dir: Path,
) -> list[dict]:
    """Scrape a list of PyTorch doc pages and save to JSON."""
    output_dir.mkdir(parents=True, exist_ok=True)
    pages = []
    for i, page_path in enumerate(page_list, 1):
        url = urljoin(base_url, page_path)
        print(f"[{i}/{len(page_list)}] {url}")
        try:
            pages.append(scrape_pytorch_doc_page(url))
            time.sleep(1)  # be nice to the server
        except Exception as e:
            print(f"  ⚠ Error: {e}")

    output_file = output_dir / "pytorch_docs.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(pages, f, indent=2, ensure_ascii=False)

    print(f"\nScraped {len(pages)} pages → {output_file}")
    return pages

In [ ]:
pytorch_pages = collect_pytorch_docs(PYTORCH_BASE_URL, PYTORCH_PAGES, PYTORCH_OUTPUT_DIR)
print(f"\n✅ PyTorch docs saved to: {PYTORCH_OUTPUT_DIR}")

In [ ]:
# Inspect
print(f"Total pages: {len(pytorch_pages)}")
for p in pytorch_pages[:3]:
    print(f"  • {p['title']}  ({len(p['content'])} chars, {len(p['code_examples'])} code blocks)")

---
# Part IV — Models

---

## 10. Prefetch Model

Download a base LLM from HuggingFace Hub.

Available presets:
- `ministral/Ministral-3b-instruct`
- `mistralai/Mistral-7B-v0.1`
- `Qwen/Qwen3-0.6B`

In [ ]:
# ---- Model configuration ----
MODEL_ID = "ministral/Ministral-3b-instruct"
MODEL_TARGET_DIR = ASSETS_DIR / "models" / MODEL_ID

In [ ]:
def download_model(model_id: str, target_dir: Path) -> Path:
    """Download a HuggingFace model repo snapshot to a local directory."""
    target_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading model repo {model_id} to {target_dir} ...")
    local_repo = snapshot_download(
        repo_id=model_id,
        local_dir=str(target_dir),
        local_dir_use_symlinks=False,
    )
    return Path(local_repo)

In [ ]:
model_path = download_model(MODEL_ID, MODEL_TARGET_DIR)
print(f"\n✅ Model stored at: {model_path}")

In [ ]:
# Inspect downloaded model files
for f in sorted(MODEL_TARGET_DIR.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.relative_to(MODEL_TARGET_DIR)}  ({size_mb:.1f} MB)")

---
# DVC — Commit Assets

After you are happy with the downloaded assets, track them with DVC:

```bash
# Training datasets
dvc add assets/datasets/arxiv-summarization-01
dvc add assets/datasets/open-code-instruct-01

# Evaluation datasets
dvc add assets/datasets/humaneval
dvc add assets/datasets/natural-questions-01
dvc add assets/datasets/hotpotqa-01
dvc add assets/datasets/msmarco-01
dvc add assets/datasets/beir-scifact
dvc add assets/datasets/beir-nfcorpus

# RAG corpora
dvc add assets/rag_data/arxiv
dvc add assets/rag_data/pytorch_docs

# Models
dvc add assets/models/ministral/Ministral-3b-instruct

dvc push
```

> **Next step (RAG only):** Build vector indices by running
> `python experiments/scripts/rag_data/build_vector_index.py`.